<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/sequence_mismatch_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence mismatch — prediction error in a learned temporal sequence (Neuropixels)

The standard-oddball paradigm (Result 1) defines "expected" by **frequency**. The sequence
paradigm defines it by **learned temporal order**: a fixed 4-element sequence
**90° → 45° → 0° → 45°** (each element 267 ms), repeated ~1250 times per session. The animal
learns the order, so the 3rd element (0°) is *predicted* by the two before it.

**Deviants** (35 each) replace that expected 3rd element with an orientation shift
(`orientation_45`, `orientation_90`), a blank (`omission`), or a flow-halt (`halt`).

**The tuning-controlled contrast (DvI).** A raw "90° deviant vs expected 0°" comparison is
confounded by orientation preference — V1 cells simply respond more to 90° than 0°. We remove
that exactly as in Result 1: compare the sequence-deviant to the **physically identical grating
presented equiprobably** in `Control block 2` (`sequential_control_block`, matched 0.25 s
duration, TF = 2 Hz), where the same orientation carries **no sequence expectation**:

    DvI = (R_deviant − R_control) / (|R_deviant| + |R_control|)

A positive DvI means the *sequential context* — not the orientation — drives the extra response.

> **Note on the data labels.** The `sequence_omission` trial type (position 5) is *always* a
> blank ending every sequence (present in 100 % of sequences), so it is the fully-predicted
> terminus, **not** a violation — its negative response is loss of visual drive, and we do not
> treat it as a prediction-error contrast.

In [ ]:
# Setup
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=True)
try:
    import pynwb, remfile, h5py, fsspec, aiohttp  # noqa
except ImportError:
    _pip("pynwb","remfile","h5py","fsspec","aiohttp","requests")
import numpy as np, pandas as pd, h5py, remfile, requests, re, pickle, time
from scipy import stats as ss
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

QUICK = True   # True: 3 sessions incl. the largest (830794), fast Colab pass. False: all 7.

In [ ]:
# NWB streaming + CCF-corrected unit->area mapping
def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])

SESSIONS=[("830794","6068eec9-d614-471c-aa33-0ab27dc66475"),
          ("830846","03973a42-cf55-476f-80d7-85bc402fa57b"),
          ("830849","fe5b6b44-12a6-4f59-a91f-953850564fb0"),
          ("830848","84b56f6e-7ade-4790-96e1-9bbc82cd5688"),
          ("830852","6f14398a-7abc-446c-a5ab-c008f52c7129"),
          ("830851","24822bc8-01d9-473f-9eef-615fa01119d9"),
          ("830795","4095a089-8cb6-4101-821c-9d93e2c2dee7")]
if QUICK:
    SESSIONS=[s for s in SESSIONS if s[0] in ("830794","830848","830846")]
print(f"{len(SESSIONS)} sessions")

In [ ]:
# Extractor: per-unit responses to sequence deviants + equiprobable controls + PSTHs
PRE,POST,BW=0.3,0.5,0.02
EDGES=np.arange(-PRE,POST+BW,BW); TCEN=EDGES[:-1]+BW/2
RESP=(0.03,0.28); BASE=(-0.1,-0.005)   # 0.25 s element response window

def seq_extract(subj,aid):
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        I=fh["intervals"]; g=I["Sequence mismatch block_presentations"]; gc=I["Control block 2_presentations"]
        TT=col(g,"TrialType"); ori=col(g,"Orientation").astype(float); tis=col(g,"TrialInSequence").astype(float); ts=g["start_time"][:]
        cori=col(gc,"Orientation").astype(float); ctf=col(gc,"TemporalFrequency").astype(float)
        cts=gc["start_time"][:]; cdur=gc["stop_time"][:]-gc["start_time"][:]
        U=fh["units"]; st=U["spike_times"][:]; sti=U["spike_times_index"][:]
        n=len(sti); qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"]; eloc=col(Eg,"location"); egroup=col(Eg,"group_name")
        dev=col(U,"device_name"); eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups}; bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(dd):
            for gn in groups:
                if dd[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={dd:d2g(dd) for dd in set(dev)}
        # extremum_channel_index is PER-PROBE; offset into stacked electrode table, clip to block length
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def wr(sp,tt,win): return (np.searchsorted(sp,tt+win[1])-np.searchsorted(sp,tt+win[0]))/(win[1]-win[0])
        def psth(sp,tt):
            if len(tt)==0: return np.full(len(TCEN),np.nan)
            lo=np.searchsorted(sp,tt.min()-PRE); hi=np.searchsorted(sp,tt.max()+POST); sp2=sp[lo:hi]
            rel=(sp2[None,:]-tt[:,None]).ravel(); rel=rel[(rel>=-PRE)&(rel<POST)]
            return np.histogram(rel,EDGES)[0]/(len(tt)*BW)
        # deviants at position 3; standard 0deg at position 3; equiprobable controls (drifting, matched orientation)
        ev={"odd90":ts[(TT=="orientation_90")&(tis==3)], "odd45":ts[(TT=="orientation_45")&(tis==3)],
            "std3":ts[(TT=="standard")&(tis==3)],
            "c90":cts[(np.abs(np.degrees(cori)-90)<5)&(np.abs(ctf-2)<0.1)&(cdur>=0.2)],
            "c45":cts[(np.abs(np.degrees(cori)-45)<5)&(np.abs(ctf-2)<0.1)&(cdur>=0.2)]}
        recs=[]; PS={k:[] for k in ev}
        for u in vis:
            sp=sp_(u); m=re.match(r"(VIS[a-z]*)(\d[a-b]?)?",u_loc[u])
            area=m.group(1) if m else u_loc[u]; layer=(m.group(2) or "") if m else ""
            row={"subject":subj,"area":area,"layer":layer,"uid":f"{subj}_{u}"}
            for k,tt in ev.items():
                row[f"R_{k}"]=np.mean(wr(sp,tt,RESP)-wr(sp,tt,BASE)) if len(tt)>=3 else np.nan
                PS[k].append(psth(sp,tt) if len(tt)>=5 else np.full(len(TCEN),np.nan))
            recs.append(row)
        return pd.DataFrame(recs),{k:np.array(v) for k,v in PS.items()}
    finally: fh.close()

In [ ]:
# Run the batch (streams NWBs; ~35-45 s/session)
DFS=[]; PSs={}; t0=time.time()
for subj,aid in SESSIONS:
    d,ps=seq_extract(subj,aid); DFS.append(d); PSs[subj]=ps
    print(f"  {subj}: {len(d)} units ({time.time()-t0:.0f}s)")
SEQ=pd.concat(DFS,ignore_index=True)
print(f"POOLED: {len(SEQ)} units, {SEQ.subject.nunique()} sessions | areas: {SEQ.area.value_counts().to_dict()}")

## Deviance index — tuning-controlled sequence prediction error

In [ ]:
def boot_ci_strat(vals,groups,n=5000,seed=0):
    rng=np.random.default_rng(seed); sess=np.unique(groups); meds=[]
    for _ in range(n):
        ii=np.concatenate([rng.choice(np.where(groups==s)[0],np.sum(groups==s),replace=True) for s in sess])
        meds.append(np.nanmedian(vals[ii]))
    return np.nanmedian(vals),np.percentile(meds,2.5),np.percentile(meds,97.5)

summ=[]
for tag,dev,ctrl in [("90","R_odd90","R_c90"),("45","R_odd45","R_c45")]:
    d=SEQ.dropna(subset=[dev,ctrl]).copy()
    d["DvI"]=(d[dev]-d[ctrl])/(d[dev].abs()+d[ctrl].abs()+1e-9)
    m,lo,hi=boot_ci_strat(d["DvI"].values,d.subject.values); p=ss.wilcoxon(d[dev]-d[ctrl]).pvalue
    summ.append((f"DvI_{tag}",m,lo,hi,p,(d.DvI>0).mean()))
    print(f"  DvI {tag}\u00b0 (deviant vs equiprobable): {m:+.3f} [{lo:+.3f},{hi:+.3f}] p={p:.1e} {(d.DvI>0).mean():.0%}pos")
# raw deviance (confounded by orientation) for comparison
for tag,dev in [("90","R_odd90"),("45","R_odd45")]:
    d=SEQ.dropna(subset=[dev,"R_std3"]); diff=(d[dev]-d.R_std3).values; m,lo,hi=boot_ci_strat(diff,d.subject.values)
    print(f"  raw deviance {tag}\u00b0 (dev - expected 0\u00b0): {m:+.3f} [{lo:+.3f},{hi:+.3f}] (orientation-confounded)")

### Figure — DvI, response dynamics, per-session consistency

In [ ]:
TC=TCEN*1000
def pool(k): return np.vstack([PSs[s][k] for s in PSs])
fig,axes=plt.subplots(1,3,figsize=(14,4.6)); fig.subplots_adjust(left=0.06,right=0.985,top=0.80,bottom=0.15,wspace=0.32)
# A: DvI forest
axA=axes[0]; rows=[]
for tag,dev,ctrl in [("90\u00b0","R_odd90","R_c90"),("45\u00b0","R_odd45","R_c45")]:
    d=SEQ.dropna(subset=[dev,ctrl]).copy(); d["DvI"]=(d[dev]-d[ctrl])/(d[dev].abs()+d[ctrl].abs()+1e-9)
    m,lo,hi=boot_ci_strat(d["DvI"].values,d.subject.values); rows.append((tag,m,lo,hi))
for i,(tag,m,lo,hi) in enumerate(rows):
    axA.plot([lo,hi],[3-i*2]*2,color="#2c3e8c",lw=2,solid_capstyle="round"); axA.plot(m,3-i*2,"o",color="#2c3e8c",ms=7)
    axA.text(hi+0.02,3-i*2,f"{m:+.2f}",va="center",fontsize=7.5,color="#2c3e8c")
axA.axvline(0,color="k",lw=0.8,ls=":"); axA.set_yticks([3,1]); axA.set_yticklabels([f"{r[0]} deviant\nDvI vs equiprobable" for r in rows],fontsize=7.5)
axA.set_ylim(-1,4); axA.set_xlim(-0.05,0.65); axA.set_xlabel("Deviance index (DvI)")
axA.set_title("A \u00b7 Sequence prediction error (tuning-controlled)\ndeviant in learned 90\u219245\u21920\u219245 sequence vs equiprobable",loc="left",fontsize=8.2)
# B: PSTH
axB=axes[1]
for k,c,lab in [("odd90","#c0392b","90\u00b0 deviant (in sequence)"),("c90","#2c3e8c","90\u00b0 equiprobable control"),("std3","#7f8c8d","expected 0\u00b0 (pos 3)")]:
    M=pool(k); m=np.nanmean(M,0); sem=np.nanstd(M,0)/np.sqrt(np.sum(~np.isnan(M[:,0])))
    ms=gaussian_filter1d(m,0.6); axB.plot(TC,ms,color=c,lw=1.8,label=lab); axB.fill_between(TC,ms-sem,ms+sem,color=c,alpha=0.18,lw=0)
axB.axvline(0,color="k",lw=0.5,ls=":"); axB.axvspan(30,280,color="#fdf0d5",zorder=0)
axB.set_xlim(-100,450); axB.set_xlabel("time from element onset (ms)"); axB.set_ylabel("firing rate (Hz)")
axB.legend(frameon=False,fontsize=6.5,loc="upper right")
axB.set_title("B \u00b7 Deviant response dynamics\ndeviant peaks later than control \u2014 an added PE component",loc="left",fontsize=8.2)
# C: per-session consistency
axC=axes[2]; sd=[]
for s in sorted(SEQ.subject.unique()):
    d=SEQ[SEQ.subject==s].dropna(subset=["R_odd90","R_c90"]).copy()
    d["DvI"]=(d.R_odd90-d.R_c90)/(d.R_odd90.abs()+d.R_c90.abs()+1e-9); sd.append((s,d.DvI.median(),len(d)))
sd.sort(key=lambda x:x[1]); ys=np.arange(len(sd))
axC.barh(ys,[x[1] for x in sd],color=["#2c3e8c" if x[1]>0 else "#c0392b" for x in sd])
axC.axvline(0,color="k",lw=0.8); axC.set_yticks(ys); axC.set_yticklabels([f"{x[0]} (n{x[2]})" for x in sd],fontsize=6.8)
axC.set_xlabel("median DvI (90\u00b0 deviant)")
axC.set_title(f"C \u00b7 Per-session consistency (90\u00b0)\n{sum(x[1]>0 for x in sd)}/{len(sd)} sessions positive",loc="left",fontsize=8.2)
fig.suptitle("Sequence mismatch \u2014 deviants in a learned temporal sequence drive a tuning-controlled prediction-error response",fontsize=9.6,y=0.955)
plt.show()

### Takeaway

- **Sequence deviants drive a positive, tuning-controlled prediction-error response**: after
  removing orientation preference (deviant vs. the same grating shown equiprobably), DvI ≈ +0.21
  (90°) and +0.35 (45°), 6/7 sessions positive. The *sequential context* — not the orientation —
  carries the extra response.
- **The dynamics show a distinct PE component**: the deviant response peaks later (~110 ms) than
  the equiprobable control (~50 ms), an added component on top of the sensory drive.
- **Same signature as the standard-oddball (Result 1)**, now with expectation set by *learned
  temporal order* rather than frequency — another error type pointing the same way, consistent
  with a common deviance-detection mechanism (H1 of `arXiv:2504.09614`).
- **Not overclaimed:** the position-5 blank is the fixed sequence terminus (not a violation), and
  the apparent suppression of the expected in-sequence 0° cannot be separated from adaptation to
  the specific preceding element, so we rest the result on the confound-controlled DvI.